# Compare PanNuke CellViT Runs: Flip Cases & Visualization

Compare inference outputs from 4 CellViT runs (baseline, z1z4, z3z4, z4), compute per-image deltas vs baseline, and visualize the biggest flips.

**Sections:**
- A) Auto-locate inference JSON files
- B) Load & parse metrics
- C) Per-image dataframe with deltas
- D) Flip case tables (best/worst by run)
- D) Compute deltas vs baseline
- E) Case selection tables (best/worst/flip)
- F) Visualization (PATH CONFIG + side-by-side viewer + dropdown)
- G) Marker selection audit

In [18]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

## Config: Run directories

In [19]:
# ----------------------------
# Run directories (defaults)
# ----------------------------
ROOT = Path("/projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis")
PANNUKE_ROOT = ROOT / "Datasets/pannuke_hf_cellvit"

RUN_BASELINE = PANNUKE_ROOT / "trainings/2026-02-20T113238_CellViTVirchowRosie-baseline-PanNuke-HF"
RUN_Z1Z4     = PANNUKE_ROOT / "trainings/2026-02-20T113238_CellViTVirchowRosieFiLM-z1z4-PanNuke-HF"
RUN_Z3Z4     = PANNUKE_ROOT / "trainings/2026-02-20T114414_CellViTVirchowRosieFiLM-z3z4-PanNuke-HF"
RUN_Z4       = PANNUKE_ROOT / "trainings/2026-02-20T122833_CellViTVirchowRosieFiLM-z4-PanNuke-HF"

RUN_DIRS: Dict[str, Path] = {
    "baseline": RUN_BASELINE,
    "z1z4":     RUN_Z1Z4,
    "z3z4":     RUN_Z3Z4,
    "z4":       RUN_Z4,
}

# Optional: manual override for JSON path per run (if auto-locate picks wrong file)
JSON_OVERRIDE: Dict[str, Optional[Path]] = {
    "baseline": None,
    "z1z4":     None,
    "z3z4":     None,
    "z4":       None,
}

top_k = 12  # Number of cases in flip tables

## A) Auto-locate inference JSON

In [20]:
def _load_json_header(path: Path, max_bytes: int = 80000) -> dict:
    """Load JSON (or header) to validate structure. For large files, peek first bytes."""
    size = path.stat().st_size
    if size < 5 * 1024 * 1024:  # < 5MB: load fully
        with open(path) as f:
            return json.load(f)
    with open(path, "r") as f:
        chunk = f.read(max_bytes)
    if '"image_metrics"' in chunk and ('"bPQ"' in chunk or '"mpq"' in chunk or '"mPQ"' in chunk):
        return {"image_metrics": {}, "dataset": {}}  # Pass validation
    return {}


def _is_inference_json(d: dict) -> bool:
    """Check if dict has structure of CellViT inference results."""
    if not isinstance(d, dict):
        return False
    has_im = "image_metrics" in d and isinstance(d.get("image_metrics"), dict)
    ds = d.get("dataset", {})
    has_pq = any(k in str(ds).lower() for k in ["mpq", "bpq", "pq"])
    return has_im and (has_pq or len(d.get("image_metrics", {})) > 0)


def find_inference_json(run_dir: Path) -> Path:
    """
    Recursively search run_dir for JSON files and select the most likely inference output.
    Heuristics:
    1) Prefer filenames containing: inference, results, pannuke, metrics
    2) Prefer JSONs with image_metrics and mpq/bPQ
    3) If multiple candidates, pick newest by mtime
    """
    run_dir = Path(run_dir)
    keywords = ["inference", "results", "pannuke", "metrics"]

    candidates: List[Tuple[Path, float, int]] = []  # (path, score, mtime)
    for p in run_dir.rglob("*.json"):
        if not p.is_file():
            continue
        name_lower = p.name.lower()
        score = sum(1 for kw in keywords if kw in name_lower)
        try:
            d = _load_json_header(p)
            if _is_inference_json(d):
                score += 10  # Strong indicator
        except Exception:
            pass
        mtime = p.stat().st_mtime
        candidates.append((p, score, int(mtime)))

    if not candidates:
        # Try direct paths
        for name in ["inference_results_normal.json", "inference_results.json"]:
            direct = run_dir / name
            if direct.exists():
                return direct
        raise FileNotFoundError(f"No inference JSON found under {run_dir}")

    # Sort by score desc, then mtime desc
    candidates.sort(key=lambda x: (-x[1], -x[2]))
    return candidates[0][0]


# Resolve and print selected JSON for each run
JSON_PATHS: Dict[str, Path] = {}
for label, rd in RUN_DIRS.items():
    if JSON_OVERRIDE.get(label):
        p = Path(JSON_OVERRIDE[label])
        if not p.exists():
            raise FileNotFoundError(f"Override path not found: {p}")
        JSON_PATHS[label] = p
        print(f"{label}: (override) {p}")
    else:
        p = find_inference_json(rd)
        JSON_PATHS[label] = p
        print(f"{label}: {p}")


baseline: /projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis/Datasets/pannuke_hf_cellvit/trainings/2026-02-20T113238_CellViTVirchowRosie-baseline-PanNuke-HF/inference_results_normal.json
z1z4: /projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis/Datasets/pannuke_hf_cellvit/trainings/2026-02-20T113238_CellViTVirchowRosieFiLM-z1z4-PanNuke-HF/inference_results_normal.json
z3z4: /projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis/Datasets/pannuke_hf_cellvit/trainings/2026-02-20T114414_CellViTVirchowRosieFiLM-z3z4-PanNuke-HF/inference_results_normal.json
z4: /projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis/Datasets/pannuke_hf_cellvit/trainings/2026-02-20T122833_CellViTVirchowRosieFiLM-z4-PanNuke-HF/inference_results_normal.json


## B) Load & parse

In [21]:
def _safe_float(x) -> float:
    try:
        if x is None:
            return float("nan")
        if isinstance(x, str) and x.lower() == "nan":
            return float("nan")
        return float(x)
    except Exception:
        return float("nan")


def _normalize_key(d: dict, *aliases: str) -> Optional[float]:
    """Get first matching key (case-insensitive)."""
    keys_lower = {k.lower(): k for k in d}
    for a in aliases:
        if a.lower() in keys_lower:
            return _safe_float(d[keys_lower[a.lower()]])
    return float("nan")


def load_inference_json(path: Path) -> dict:
    with open(path, "r") as f:
        return json.load(f)


# Load all JSONs
RESULTS: Dict[str, dict] = {label: load_inference_json(p) for label, p in JSON_PATHS.items()}

# Global metrics table
GLOBAL_KEYS = ["mPQ", "mDQ", "mSQ", "bPQ", "bDQ", "bSQ", "Binary-Cell-Dice-Mean", "f1_detection", "precision_detection", "recall_detection", "Tissue-Multiclass-Accuracy"]
global_rows = []
for label, d in RESULTS.items():
    ds = d.get("dataset", {})
    row = {"run": label}
    for k in GLOBAL_KEYS:
        row[k] = _normalize_key(ds, k, k.lower().replace("-", "_"))
    global_rows.append(row)

global_df = pd.DataFrame(global_rows).set_index("run")
display(global_df.round(4))

,mPQ,mDQ,mSQ,bPQ,bDQ,bSQ,Binary-Cell-Dice-Mean,f1_detection,precision_detection,recall_detection,Tissue-Multiclass-Accuracy
run,,,,,,,,,,,
baseline,0.7589,0.8524,0.8406,0.8155,0.9163,0.8840,0.8992,0.9296,0.9564,0.9043,0.9960
z1z4,0.7287,0.8233,0.8270,0.8005,0.9044,0.8789,0.8905,0.9169,0.9394,0.8954,0.9945
z3z4,0.7470,0.8400,0.8411,0.8094,0.9098,0.8842,0.8939,0.9192,0.9357,0.9033,0.9993
z4,0.7539,0.8529,0.8383,0.8077,0.9138,0.8773,0.8961,0.9252,0.9444,0.9067,0.9978


## C) Per-image dataframe

In [22]:
# Build per-image dataframe: image_id x (baseline_pq, z4_pq, ... baseline_bpq, ... baseline_dice, ...)
# PanNuke inference stores per-image: Dice, Jaccard, bPQ (no mPQ/dq/sq per image)
# We use bPQ as primary 'pq' and include Dice

def _get_im_metric(im: dict, *keys: str) -> float:
    return _normalize_key(im, *keys)


all_ids = set()
for d in RESULTS.values():
    all_ids.update(d.get("image_metrics", {}).keys())
all_ids = sorted(all_ids)

rows = []
for img_id in all_ids:
    row = {"image_id": img_id}
    for label, d in RESULTS.items():
        im = d.get("image_metrics", {}).get(img_id, {})
        row[f"{label}_pq"]   = _get_im_metric(im, "bPQ", "bpq", "mPQ", "mpq", "pq")
        row[f"{label}_bpq"]  = _get_im_metric(im, "bPQ", "bpq")
        row[f"{label}_dice"] = _get_im_metric(im, "Dice", "dice")
    rows.append(row)

df = pd.DataFrame(rows).set_index("image_id")

# Load tissue type from PanNuke types.csv if available
TISSUE_MAP: Dict[str, str] = {}
for fold in [2]:  # test fold
    types_path = PANNUKE_ROOT / f"fold{fold}" / "types.csv"
    if types_path.exists():
        tf = pd.read_csv(types_path)
        for _, r in tf.iterrows():
            TISSUE_MAP[str(r.get("img", r.get("image_id", "")))] = str(r.get("type", ""))

if TISSUE_MAP:
    df["tissue_type"] = df.index.map(lambda x: TISSUE_MAP.get(x, ""))

print(f"Per-image dataframe: {len(df)} images")
df.head(10)

Per-image dataframe: 2722 images


,baseline_pq,baseline_bpq,baseline_dice,z1z4_pq,z1z4_bpq,z1z4_dice,z3z4_pq,z3z4_bpq,z3z4_dice,z4_pq,z4_bpq,z4_dice,tissue_type
image_id,,,,,,,,,,,,,
000000.png,0.908257,0.908257,0.961339,0.899756,0.899756,0.957574,0.911825,0.911825,0.963752,0.909292,0.909292,0.962201,Breast
000001.png,0.911737,0.911737,0.963614,0.914336,0.914336,0.966289,0.898612,0.898612,0.960306,0.918134,0.918134,0.963701,Breast
000002.png,0.685957,0.685957,0.952081,0.819179,0.819179,0.939763,0.797011,0.797011,0.932740,0.718441,0.718441,0.942250,Breast
000003.png,0.895881,0.895881,0.965048,0.753910,0.753910,0.954339,0.881317,0.881317,0.966097,0.883923,0.883923,0.963055,Breast
000004.png,0.909575,0.909575,0.980934,0.888405,0.888405,0.973932,0.909403,0.909403,0.978984,0.905833,0.905833,0.979236,Breast
000005.png,0.774343,0.774343,0.969663,0.791793,0.791793,0.969032,0.795585,0.795585,0.970784,0.807790,0.807790,0.966293,Breast
000006.png,0.804170,0.804170,0.967791,0.745624,0.745624,0.963916,0.799564,0.799564,0.968413,0.802692,0.802692,0.965161,Breast
000007.png,0.930662,0.930662,0.967506,0.919973,0.919973,0.959616,0.929832,0.929832,0.967173,0.913026,0.913026,0.962120,Breast
000008.png,0.875945,0.875945,0.960811,0.851522,0.851522,0.956410,0.852416,0.852416,0.960195,0.885813,0.885813,0.963052,Breast


## D) Compute deltas vs baseline

In [23]:
# Deltas (FiLM run - baseline)
for run in ["z4", "z3z4", "z1z4"]:
    if f"{run}_pq" in df.columns and "baseline_pq" in df.columns:
        df[f"delta_{run}_pq"]   = df[f"{run}_pq"]   - df["baseline_pq"]
        df[f"delta_{run}_bpq"]  = df[f"{run}_bpq"]  - df["baseline_bpq"]
        df[f"delta_{run}_dice"] = df[f"{run}_dice"] - df["baseline_dice"]

# Flip score: max absolute delta across FiLM runs
delta_cols = [c for c in df.columns if c.startswith("delta_") and "_pq" in c]
if delta_cols:
    df["flip_score"] = df[delta_cols].abs().max(axis=1)

df.head(10)

,baseline_pq,baseline_bpq,baseline_dice,z1z4_pq,z1z4_bpq,z1z4_dice,z3z4_pq,z3z4_bpq,z3z4_dice,z4_pq,z4_bpq,z4_dice,tissue_type,delta_z4_pq,delta_z4_bpq,delta_z4_dice,delta_z3z4_pq,delta_z3z4_bpq,delta_z3z4_dice,delta_z1z4_pq,delta_z1z4_bpq,delta_z1z4_dice,flip_score
image_id,,,,,,,,,,,,,,,,,,,,,,,
000000.png,0.908257,0.908257,0.961339,0.899756,0.899756,0.957574,0.911825,0.911825,0.963752,0.909292,0.909292,0.962201,Breast,0.001035,0.001035,0.000863,0.003568,0.003568,0.002413,-0.008501,-0.008501,-0.003764,0.008501
000001.png,0.911737,0.911737,0.963614,0.914336,0.914336,0.966289,0.898612,0.898612,0.960306,0.918134,0.918134,0.963701,Breast,0.006397,0.006397,0.000087,-0.013125,-0.013125,-0.003308,0.002599,0.002599,0.002675,0.013125
000002.png,0.685957,0.685957,0.952081,0.819179,0.819179,0.939763,0.797011,0.797011,0.932740,0.718441,0.718441,0.942250,Breast,0.032484,0.032484,-0.009832,0.111054,0.111054,-0.019341,0.133222,0.133222,-0.012319,0.133222
000003.png,0.895881,0.895881,0.965048,0.753910,0.753910,0.954339,0.881317,0.881317,0.966097,0.883923,0.883923,0.963055,Breast,-0.011959,-0.011959,-0.001992,-0.014564,-0.014564,0.001050,-0.141971,-0.141971,-0.010709,0.141971
000004.png,0.909575,0.909575,0.980934,0.888405,0.888405,0.973932,0.909403,0.909403,0.978984,0.905833,0.905833,0.979236,Breast,-0.003742,-0.003742,-0.001699,-0.000172,-0.000172,-0.001950,-0.021170,-0.021170,-0.007003,0.021170
000005.png,0.774343,0.774343,0.969663,0.791793,0.791793,0.969032,0.795585,0.795585,0.970784,0.807790,0.807790,0.966293,Breast,0.033446,0.033446,-0.003370,0.021241,0.021241,0.001121,0.017449,0.017449,-0.000631,0.033446
000006.png,0.804170,0.804170,0.967791,0.745624,0.745624,0.963916,0.799564,0.799564,0.968413,0.802692,0.802692,0.965161,Breast,-0.001477,-0.001477,-0.002630,-0.004606,-0.004606,0.000622,-0.058546,-0.058546,-0.003875,0.058546
000007.png,0.930662,0.930662,0.967506,0.919973,0.919973,0.959616,0.929832,0.929832,0.967173,0.913026,0.913026,0.962120,Breast,-0.017636,-0.017636,-0.005387,-0.000830,-0.000830,-0.000333,-0.010689,-0.010689,-0.007890,0.017636
000008.png,0.875945,0.875945,0.960811,0.851522,0.851522,0.956410,0.852416,0.852416,0.960195,0.885813,0.885813,0.963052,Breast,0.009868,0.009868,0.002241,-0.023528,-0.023528,-0.000616,-0.024423,-0.024423,-0.004401,0.024423


## E) Case selection (flip tables)

In [24]:
def _styled_table(df: pd.DataFrame, title: str = "") -> pd.io.formats.style.Styler:
    """Clean styled table: 4 decimals, highlight improvements (green) and regressions (red)."""
    sty = df.style.format(lambda x: f"{x:+.4f}" if isinstance(x, (int, float)) and "delta" in str(df.columns.tolist()) else f"{x:.4f}" if isinstance(x, (int, float)) else str(x))
    return sty.set_caption(title)


def _make_flip_table(df: pd.DataFrame, sort_col: str, ascending: bool, k: int) -> pd.DataFrame:
    cols = ["image_id", sort_col]
    for c in ["baseline_pq", "z4_pq", "z3z4_pq", "z1z4_pq"]:
        if c in df.columns:
            cols.append(c)
    if "tissue_type" in df.columns:
        cols.append("tissue_type")
    cols = [c for c in cols if c in df.columns]
    return df.nsmallest(k, sort_col) if ascending else df.nlargest(k, sort_col)[cols]


display_cols = ["image_id", "delta_z4_pq", "baseline_pq", "z4_pq", "z3z4_pq", "z1z4_pq"]
display_cols = [c for c in display_cols if c in df.columns]
if "tissue_type" in df.columns:
    display_cols.append("tissue_type")


In [25]:
print("1) Best improvements for z4 (top by delta_z4_pq)")
t1 = df.nlargest(top_k, "delta_z4_pq")[display_cols]
display(t1)

1) Best improvements for z4 (top by delta_z4_pq)


,delta_z4_pq,baseline_pq,z4_pq,z3z4_pq,z1z4_pq,tissue_type
image_id,,,,,,
002073.png,0.568524,0.294207,0.862731,0.905453,0.871760,Head & Neck
002656.png,0.352941,0.000000,0.352941,0.000000,0.414239,Colon
000593.png,0.316818,0.632171,0.948989,0.634027,0.923157,Breast
002068.png,0.311560,0.462744,0.774305,0.656400,0.307692,Head & Neck
001051.png,0.304532,0.430296,0.734828,0.367493,0.550653,Esophagus
001878.png,0.254025,0.610441,0.864465,0.624483,0.624649,Colon
002637.png,0.222340,0.505293,0.727633,0.487444,0.490687,Colon
000740.png,0.214353,0.687129,0.901483,0.905266,0.915857,Colon
001784.png,0.211690,0.707519,0.919209,0.910788,0.885348,Colon


In [26]:
print("2) Worst regressions for z4 (bottom by delta_z4_pq)")
t2 = df.nsmallest(top_k, "delta_z4_pq")[display_cols]
display(t2)

2) Worst regressions for z4 (bottom by delta_z4_pq)


,delta_z4_pq,baseline_pq,z4_pq,z3z4_pq,z1z4_pq,tissue_type
image_id,,,,,,
000550.png,-0.598957,0.598957,0.000000,0.000000,0.000000,Breast
001469.png,-0.545237,0.545237,0.000000,0.579016,0.604081,Bile Duct
002685.png,-0.477777,0.477777,0.000000,0.595959,0.000000,Colon
000758.png,-0.451461,0.451461,0.000000,0.488888,0.000000,Colon
002175.png,-0.281382,0.896058,0.614676,0.930435,0.928878,Liver
000924.png,-0.279236,0.875138,0.595902,0.778624,0.844843,Esophagus
002693.png,-0.277182,0.549439,0.272258,0.442086,0.281453,Colon
001470.png,-0.247680,0.911202,0.663522,0.905663,0.879058,Bile Duct
001403.png,-0.235491,0.711183,0.475692,0.698091,0.570440,Adrenal Gland


In [27]:
print("3) Best / worst for z3z4")
t3a = df.nlargest(top_k, "delta_z3z4_pq")[display_cols]
t3b = df.nsmallest(top_k, "delta_z3z4_pq")[display_cols]
display(t3a)
display(t3b)

3) Best / worst for z3z4


,delta_z4_pq,baseline_pq,z4_pq,z3z4_pq,z1z4_pq,tissue_type
image_id,,,,,,
001612.png,0.000000,0.000000,0.000000,0.646550,0.757280,Breast
002073.png,0.568524,0.294207,0.862731,0.905453,0.871760,Head & Neck
001086.png,0.000000,0.000000,0.000000,0.366492,0.362416,Cervix
002535.png,0.006350,0.573896,0.580246,0.939257,0.929795,Uterus
001929.png,0.090596,0.634003,0.724600,0.898442,0.828821,Colon
001476.png,0.123242,0.334150,0.457392,0.584563,0.579491,Bile Duct
002144.png,0.105611,0.582052,0.687663,0.830986,0.686588,Kidney
002696.png,0.084978,0.546816,0.631794,0.783661,0.599999,Colon
001364.png,-0.026048,0.267806,0.241758,0.499350,0.330434,Colon


,delta_z4_pq,baseline_pq,z4_pq,z3z4_pq,z1z4_pq,tissue_type
image_id,,,,,,
001466.png,0.025677,0.608937,0.634614,0.000000,0.654793,Adrenal Gland
000550.png,-0.598957,0.598957,0.000000,0.000000,0.000000,Breast
000156.png,-0.023221,0.933331,0.910111,0.596106,0.919353,Breast
000430.png,-0.008651,0.774379,0.765728,0.473702,0.598032,Breast
001089.png,-0.041619,0.895889,0.854270,0.597700,0.811413,Cervix
000783.png,0.047942,0.917681,0.965623,0.632007,0.952828,Colon
000357.png,-0.001838,0.808641,0.806803,0.537514,0.846693,Breast
002688.png,0.081818,0.458660,0.540478,0.230919,0.478183,Colon
001916.png,-0.000814,0.872111,0.871297,0.644540,0.817820,Colon


In [28]:
print("4) Top flip_score cases (largest absolute deltas across all runs)")
t4 = df.nlargest(top_k, "flip_score")[display_cols + ["flip_score"] if "flip_score" in df.columns else display_cols]
display(t4)

4) Top flip_score cases (largest absolute deltas across all runs)


,delta_z4_pq,baseline_pq,z4_pq,z3z4_pq,z1z4_pq,tissue_type,flip_score
image_id,,,,,,,
001405.png,0.006590,0.872926,0.879516,0.868851,0.000000,Adrenal Gland,0.872926
001612.png,0.000000,0.000000,0.000000,0.646550,0.757280,Breast,0.757280
002073.png,0.568524,0.294207,0.862731,0.905453,0.871760,Head & Neck,0.611246
001466.png,0.025677,0.608937,0.634614,0.000000,0.654793,Adrenal Gland,0.608937
000550.png,-0.598957,0.598957,0.000000,0.000000,0.000000,Breast,0.598957
001469.png,-0.545237,0.545237,0.000000,0.579016,0.604081,Bile Duct,0.545237
002071.png,0.040469,0.491582,0.532050,0.541253,0.000000,Head & Neck,0.491582
002685.png,-0.477777,0.477777,0.000000,0.595959,0.000000,Colon,0.477777
000758.png,-0.451461,0.451461,0.000000,0.488888,0.000000,Colon,0.451461


In [29]:
# If tissue_type exists: lung subset + mean deltas
if "tissue_type" in df.columns:
    df_lung = df[df["tissue_type"].str.lower() == "lung"]
    df_nonlung = df[df["tissue_type"].str.lower() != "lung"]
    if len(df_lung) > 0:
        print("5a) Lung: best/worst z4 deltas")
        t5a = df_lung.nlargest(top_k, "delta_z4_pq")[display_cols]
        t5b = df_lung.nsmallest(top_k, "delta_z4_pq")[display_cols]
        display(t5a)
        display(t5b)
        print("5b) Mean deltas: lung vs non-lung")
        delta_cols = [c for c in df.columns if c.startswith("delta_") and "_pq" in c]
        means = pd.DataFrame({
            "lung": df_lung[delta_cols].mean(),
            "non_lung": df_nonlung[delta_cols].mean(),
        })
        display(means.round(4))
    else:
        print("No lung tissue in test set.")
else:
    print("Tissue type not available; skipping lung-specific tables.")

5a) Lung: best/worst z4 deltas


,delta_z4_pq,baseline_pq,z4_pq,z3z4_pq,z1z4_pq,tissue_type
image_id,,,,,,
000953.png,0.057823,0.721133,0.778956,0.722517,0.777750,Lung
000938.png,0.055126,0.498747,0.553874,0.574919,0.604638,Lung
000941.png,0.039994,0.640477,0.680470,0.687693,0.679089,Lung
000935.png,0.036842,0.677806,0.714648,0.742546,0.726327,Lung
000932.png,0.031192,0.555429,0.586621,0.560851,0.584126,Lung
000829.png,0.030749,0.856516,0.887265,0.860221,0.880322,Lung
000823.png,0.027872,0.845864,0.873736,0.865500,0.840003,Lung
000929.png,0.027736,0.653203,0.680939,0.739754,0.702943,Lung
000828.png,0.027723,0.787177,0.814900,0.830625,0.774013,Lung


,delta_z4_pq,baseline_pq,z4_pq,z3z4_pq,z1z4_pq,tissue_type
image_id,,,,,,
002260.png,-0.077957,0.825465,0.747509,0.823719,0.793891,Lung
000949.png,-0.057266,0.709110,0.651844,0.730048,0.731949,Lung
000945.png,-0.056104,0.800927,0.744823,0.750770,0.750613,Lung
000955.png,-0.053270,0.826942,0.773672,0.831242,0.840781,Lung
002263.png,-0.046379,0.864969,0.818590,0.798920,0.829710,Lung
000818.png,-0.042376,0.803303,0.760928,0.691799,0.759160,Lung
000951.png,-0.035062,0.847809,0.812747,0.792603,0.790642,Lung
000954.png,-0.033532,0.905402,0.871870,0.918578,0.914964,Lung
000947.png,-0.028463,0.872356,0.843893,0.873516,0.832906,Lung


5b) Mean deltas: lung vs non-lung


,lung,non_lung
delta_z4_pq,-0.0032,-0.0079
delta_z3z4_pq,-0.0012,-0.0062
delta_z1z4_pq,-0.0014,-0.0152


## F) PATH CONFIG for visualization

**Why are baseline/z1z4/z3z4/z4 prediction columns empty?**

Prediction images are only saved when inference is run with `--plots`. 

**Option A – Plots for high-delta cases only (faster):** Run the case-selection cell to export `flip_cases.txt`, then run inference with `--plot_image_ids` for each run:

```bash
python .../inference_cellvit_experiment_pannuke.py --run_dir /path/to/run --plots --plot_image_ids flip_cases.txt --plots_only
```

This infers only the listed images and saves plots in `inference_predictions/` without overwriting the full inference JSON.

**Option B – Full plots:** Re-run full inference with `--plots` (slow, all images).

In [30]:
# ----------------------------
# PATH CONFIG: Patch images, GT masks, predictions
# ----------------------------
# PanNuke standard: dataset_path/fold2/images/*.png, fold2/labels/{stem}.npy
# Predictions: typically run_dir/inference_predictions/ (if generate_plots=True during inference)
# Fill these or leave None to skip visualization.

TEST_FOLD = 2
PATCH_DIR = PANNUKE_ROOT / f"fold{TEST_FOLD}/images"   # RGB patch images
GT_DIR    = PANNUKE_ROOT / f"fold{TEST_FOLD}/labels"   # GT .npy (instance + type)

# Per-run prediction dirs (instance/type masks or overlay images)
PRED_DIRS: Dict[str, Optional[Path]] = {
    "baseline": RUN_BASELINE / "inference_predictions" if (RUN_BASELINE / "inference_predictions").exists() else None,
    "z1z4":     RUN_Z1Z4     / "inference_predictions" if (RUN_Z1Z4     / "inference_predictions").exists() else None,
    "z3z4":     RUN_Z3Z4     / "inference_predictions" if (RUN_Z3Z4     / "inference_predictions").exists() else None,
    "z4":       RUN_Z4       / "inference_predictions" if (RUN_Z4       / "inference_predictions").exists() else None,
}

VIZ_AVAILABLE = PATCH_DIR.exists() and GT_DIR.exists()
print(f"PATCH_DIR exists: {PATCH_DIR.exists()}")
print(f"GT_DIR exists: {GT_DIR.exists()}")
for k, v in PRED_DIRS.items():
    print(f"  pred {k}: {v.exists() if v else False}")

PATCH_DIR exists: True
GT_DIR exists: True
  pred baseline: False
  pred z1z4: False
  pred z3z4: False
  pred z4: False


## F2) Side-by-side viewer

In [31]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib

def _load_patch(img_id: str) -> Optional[np.ndarray]:
    if not PATCH_DIR or not PATCH_DIR.exists():
        return None
    stem = str(img_id).replace(".png", "")
    for name in [img_id, f"{stem}.png"]:
        p = PATCH_DIR / name
        if p.exists():
            from PIL import Image
            arr = np.array(Image.open(p).convert("RGB"))
            return arr
    return None


def _load_gt_mask(img_id: str) -> Optional[Tuple[np.ndarray, np.ndarray]]:
    if not GT_DIR or not GT_DIR.exists():
        return None
    stem = str(img_id).replace(".png", "")
    p = GT_DIR / f"{stem}.npy"
    if not p.exists():
        return None
    try:
        m = np.load(p, allow_pickle=True)
        # PanNuke: 0-d array with dict {inst_map, type_map}
        if isinstance(m, np.ndarray) and m.ndim == 0:
            d = m.item()
            if isinstance(d, dict):
                inst = d.get("inst_map", np.zeros((256, 256)))
                typ = d.get("type_map", np.zeros_like(inst))
                return np.asarray(inst, dtype=np.int32), np.asarray(typ, dtype=np.int32)
        if m.ndim == 3:
            inst = m[..., 0] if m.shape[-1] >= 1 else m[:, :, 0]
            typ = m[..., 1] if m.shape[-1] >= 2 else np.zeros_like(inst)
        else:
            inst = m
            typ = np.zeros_like(inst)
        return inst.astype(np.int32), typ.astype(np.int32)
    except Exception:
        return None


def _mask_to_rgb(inst: np.ndarray, type_map: Optional[np.ndarray] = None) -> np.ndarray:
    """Convert instance mask to RGB for display."""
    
    n = inst.max() if inst.size else 0
    if n == 0:
        return np.zeros((*inst.shape, 3), dtype=np.uint8)
    cmap = matplotlib.colormaps.get_cmap("tab20").resampled(max(20, n + 1))
    rgb = (cmap((inst % 20) / 19.0)[:, :, :3] * 255).astype(np.uint8)
    rgb[inst == 0] = 0
    return rgb


def show_case(image_id: str, figsize: Tuple[int, int] = (14, 4)) -> None:
    """Display: GT | baseline pred | z1z4 pred | z3z4 pred | z4 pred."""
    ncols = 6  # Image, GT, baseline, z1z4, z3z4, z4
    fig, axes = plt.subplots(1, ncols, figsize=(16, 4))
    titles = ["GT", "baseline", "z1z4", "z3z4", "z4"]
    for ax in axes:
        ax.set_xticks([])
        ax.set_yticks([])

    patch = _load_patch(image_id)
    gt = _load_gt_mask(image_id)

    # Column 0: RGB patch or GT overlay
    if patch is not None:
        axes[0].imshow(patch)
    axes[0].set_title("Image")

    # Column 1: GT mask
    if gt is not None:
        rgb_gt = _mask_to_rgb(gt[0], gt[1])
        axes[1].imshow(rgb_gt)
    axes[1].set_title("GT")

    # Columns 2-5: Predictions (if available)
    for i, (run, pred_dir) in enumerate(PRED_DIRS.items()):
        ax = axes[2 + i]
        if pred_dir and pred_dir.exists():
            stem = str(image_id).replace(".png", "")
            cands = list(pred_dir.glob(f"*{stem}*.png")) + list(pred_dir.glob(f"*{image_id}*"))
            if cands:
                arr = np.array(plt.imread(cands[0]))
                if arr.ndim == 2:
                    arr = _mask_to_rgb(arr)
                ax.imshow(arr)
        ax.set_title(run)

    plt.suptitle(f"{image_id}", fontsize=10)
    plt.tight_layout()
    plt.show()

In [32]:
# Build case list for dropdown (flip cases + optionally lung)
FLIP_CASES = df.nlargest(top_k * 2, "flip_score").index.tolist() if "flip_score" in df.columns else df.index.tolist()[:top_k * 2]
if "tissue_type" in df.columns:
    lung_cases = df[df["tissue_type"].str.lower() == "lung"].nlargest(top_k, "flip_score").index.tolist() if "flip_score" in df.columns else []
    CASE_OPTIONS = [(f"{x} (Δ={df.loc[x, 'flip_score']:.3f})" if x in df.index and 'flip_score' in df.columns else x, x) for x in (FLIP_CASES + lung_cases)]
else:
    CASE_OPTIONS = [(f"{x} (flip={df.loc[x, 'flip_score']:.3f})" if x in df.index and 'flip_score' in df.columns else x, x) for x in FLIP_CASES]

# Deduplicate while preserving order
seen = set()
CASE_OPTIONS = [(lbl, id) for lbl, id in CASE_OPTIONS if id not in seen and not seen.add(id)]

if not CASE_OPTIONS:
    CASE_OPTIONS = [(str(i), i) for i in df.index.tolist()[:20]]

print(f"Case options: {len(CASE_OPTIONS)}")

# Export high-delta image IDs for --plot_image_ids (avoids full re-inference)
EXPORT_FLIP_IDS = True  # set False to skip
FLIP_IDS_FILE = ROOT / "notebooks-test-set" / "flip_cases.txt"
if EXPORT_FLIP_IDS and CASE_OPTIONS:
    ids_to_export = list(dict.fromkeys(cid for _, cid in CASE_OPTIONS))
    FLIP_IDS_FILE.parent.mkdir(parents=True, exist_ok=True)
    FLIP_IDS_FILE.write_text("\n".join(ids_to_export))
    print(f"Exported {len(ids_to_export)} image IDs to {FLIP_IDS_FILE}")

Case options: 36


In [ ]:
# ipywidgets dropdown to browse cases
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    dropdown = widgets.Dropdown(
        options=CASE_OPTIONS,
        value=CASE_OPTIONS[0][1] if CASE_OPTIONS else None,
        description="Case:",
        layout=widgets.Layout(width="500px"),
    )
    output = widgets.Output()

    def on_select(change):
        with output:
            clear_output(wait=True)
            if change["new"]:
                show_case(change["new"])

    dropdown.observe(on_select, names="value")
    display(widgets.VBox([dropdown, output]))
    if CASE_OPTIONS:
        on_select({"new": CASE_OPTIONS[0][1]})
except ImportError:
    print("ipywidgets not installed. Call show_case(image_id) manually.")
    if CASE_OPTIONS:
        show_case(CASE_OPTIONS[0][1])

## G) Marker selection audit

In [34]:
def _inspect_config(run_dir: Path) -> dict:
    """Inspect run_dir for config (yaml/json) and extract marker/conditioning info."""
    out = {"run_dir": str(run_dir), "num_markers": None, "marker_names": [], "conditioning": "none"}
    configs = list(run_dir.rglob("config*.yaml")) + list(run_dir.rglob("config*.yml")) + list(run_dir.rglob("config*.json"))
    for p in configs:
        if not p.is_file() or p.stat().st_size > 500_000:
            continue
        try:
            text = p.read_text()
            if p.suffix in [".yaml", ".yml"]:
                cfg = __import__("yaml").safe_load(text)
            else:
                cfg = json.loads(text)
            if not isinstance(cfg, dict):
                continue
            # Fusion / model config (nested)
            for root in [cfg, cfg.get("config", {}), cfg.get("model", {}), cfg.get("fusion", {})]:
                if not isinstance(root, dict):
                    continue
                if "film_enabled" in root and root.get("film_enabled"):
                    out["conditioning"] = "FiLM"
                if "film_layers" in root and root.get("film_layers"):
                    out["conditioning"] = "FiLM"
                if "early_fusion" in str(root).lower() or "earlyfusion" in str(root).lower():
                    out["conditioning"] = "early fusion"
                markers = root.get("markers") or root.get("marker_names") or root.get("rosie_marker_subset")
                if markers:
                    out["marker_names"] = list(markers)[:10] if isinstance(markers, (list, tuple)) else [str(markers)]
                    out["num_markers"] = len(markers) if isinstance(markers, (list, tuple)) else None
                nm = root.get("num_markers") or root.get("channels")
                if nm is not None and out["num_markers"] is None:
                    out["num_markers"] = int(nm)
        except Exception:
            pass
    return out


audit_rows = []
for label, rd in RUN_DIRS.items():
    info = _inspect_config(rd)
    audit_rows.append({
        "run": label,
        "num_markers": info["num_markers"],
        "marker_names_preview": ", ".join(str(x) for x in info["marker_names"][:5]) if info["marker_names"] else "—",
        "conditioning": info["conditioning"],
    })

audit_df = pd.DataFrame(audit_rows)
print("Marker selection audit:")
display(audit_df)

Marker selection audit:


,run,num_markers,marker_names_preview,conditioning
0,baseline,None,—,FiLM
1,z1z4,None,—,FiLM
2,z3z4,None,—,FiLM
3,z4,None,—,FiLM
